# Deploy a scikit-learn model on TrueFoundry with FastAPI, no inference code

- Flow: **train → log model → package → build & push image → deploy → infer**
- Inference code: autogenerated by TFY CLI
- Image: build and pushed to registry first, never built by TrueFoundry
- Docs: https://www.truefoundry.com/docs/model-deployment/deploy-sklearn-xgboost

---

## 0. Setup

> **Kernel: `Python 3`** — the default env (Python 3.14).

In [1]:
# %pip install -U truefoundry scikit-learn joblib requests

In [2]:
import sys
import time
from pathlib import Path

import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

from truefoundry.ml import get_client, SklearnFramework, sklearn_infer_schema
from truefoundry.common.session import Session
from truefoundry_sdk import TrueFoundry

In [3]:
TFY_HOST = "https://internal.devtest.truefoundry.tech"
ML_REPO = "default"
CLUSTER = "tfy-usea1-devtest"
WORKSPACE = "seldon-tf"
DOMAIN = "tfy-usea1-ctl.devtest.truefoundry.tech"
WS_FQN = f"{CLUSTER}:{WORKSPACE}"

DOCKER_USER = "mkozubal"
TAG = "1"
PY_VER = "3.14"
PORT = 8000
READY_TIMEOUT = 300

NAME = "iris-sk-fastapi"
MODEL_FILE = f"{NAME}.joblib"
SVC = f"{NAME}-svc"
IMAGE = f"{DOCKER_USER}/{SVC}:{TAG}"
HOST = f"{SVC}-{WORKSPACE}-{PORT}.{DOMAIN}"

## 1. Train and save

In [4]:
X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=10).fit(X, y)
joblib.dump(model, MODEL_FILE)

['iris-sk-fastapi.joblib']

## 2. Log model 
- schema goes here
- `model_filepath` + `serialization_format` + `model_schema` are what make the version deployable

In [5]:
client = get_client()

framework = SklearnFramework(
    model_filepath=MODEL_FILE,
    serialization_format="joblib",
    model_schema=sklearn_infer_schema(
        model_input=X, model=model, infer_method_name="predict",
    ),
)

FQN = client.log_model(
    ml_repo=ML_REPO, name=NAME, model_file_or_folder=MODEL_FILE, framework=framework,
).fqn
print(FQN)

[truefoundry.ml] 2026-09-29T12:37:56+0100 INFO Logged in to 'https://internal.devtest.truefoundry.tech' as 'maciej.kozubal@truefoundry.com'
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T12:37:56+0100 INFO Logging model and additional files, this might take a while ...
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T12:37:56+0100 INFO Adding model file/folder to model version content
[truefoundry.ml.log_types.artifacts.utils] 2026-09-29T12:37:56+0100 INFO Adding file iris-sk-fastapi.joblib as /iris-sk-fastapi.joblib
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T12:37:57+0100 INFO Packaging and uploading files to remote with Total Size: 0.020865 MB
[truefoundry.ml] 2026-09-29T12:37:57+0100 WARNING Detected Jupyter Environment. Install `ipywidgets` to display live progress bars.
[truefoundry.ml] 2026-09-29T12:37:57+0100 INFO Uploading /Users/maciej/work/seldon-tf/tf-workloads/core2-iris-no-inference-code/notebooks/iris-sk-fastapi.joblib to iris-sk-fastapi.joblib
[tru

## 3. Generate the serving package

`tfy deploy-init` writes `server.py`, `infer.py`, `requirements.txt`, `deploy.py`, `README.md` locally.
Nothing is built and nothing is deployed — same command the UI Deploy button hands you.

Called as `{sys.executable} -m truefoundry.cli`: `!` inherits the notebook process PATH, which
does not contain the kernel venv's `bin/` unless Jupyter was started from an activated env.

In [7]:
!{sys.executable} -m truefoundry.cli deploy-init model --name {SVC} --model-version-fqn '{FQN}' --workspace-fqn '{WS_FQN}' --model-server fastapi
assert _exit_code == 0, "deploy-init failed"

[truefoundry.ml] 2026-09-29T12:38:28+0100 INFO Logged in to 'https://internal.devtest.truefoundry.tech' as 'maciej.kozubal@truefoundry.com'
Generating application code for 'model:truefoundry/default/iris-sk-fastapi:3'

Model Server code initialized successfully!

Code Location: /Users/maciej/work/seldon-tf/tf-workloads/core2-iris-no-inference-code/notebooks/iris-sk-fastapi-svc

Next Steps:
- Navigate to the model server directory:
cd /Users/maciej/work/seldon-tf/tf-workloads/core2-iris-no-inference-code/notebooks/iris-sk-fastapi-svc
- Refer to the README file in the directory for further instructions.



In [10]:
!tree {SVC}

iris-sk-fastapi-svc
├── deploy.py
├── infer.py
├── README.md
├── requirements.txt
└── server.py

1 directory, 5 files


## 4. Build and push the image

The FastAPI codegen emits **no** Dockerfile (its `deploy.py` uses `PythonBuild`, built on
TrueFoundry's builder). We add our own and push to Docker Hub instead — `deploy.py` goes unused.

In [8]:
Path(SVC, "Dockerfile").write_text(f'''FROM python:{PY_VER}-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "{PORT}"]
''')

188

In [9]:
!docker build --platform linux/amd64 -t {IMAGE} {SVC}
assert _exit_code == 0, "docker build failed"

!docker push {IMAGE}
assert _exit_code == 0, "docker push failed"



[+] Building 0.0s (0/1)                                           docker:colima
[+] Building 0.2s (1/2)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 227B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.14-slim        0.2s
[+] Building 0.3s (1/3)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 227B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.14-slim        0.3s
 => [auth] library/python:pull token for registry-1.docker.io              0.0s
[+] Building 0.5s (2/3)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 227B  

## 5. Deploy as a service

- image: prebuilt (`"image": {"type": "image", ...}`)
- `artifacts_download`: pulls the model version into `$MODEL_PATH` (the image is model-free)
- auth reuses the `tfy login` session (no PAT in the notebook)
- `create_or_update`: returns a deployment, so delete later with `app.data.application_id`

In [14]:
tfy = TrueFoundry(base_url=TFY_HOST, api_key=Session.new().access_token)

app = tfy.applications.create_or_update(manifest={
    "name": SVC, "type": "service", "workspace_fqn": WS_FQN,
    "image": {"type": "image", "image_uri": IMAGE},
    "artifacts_download": {"artifacts": [{"type": "truefoundry-artifact",
        "artifact_version_fqn": FQN, "download_path_env_variable": "MODEL_PATH"}]},
    "ports": [{"port": PORT, "protocol": "TCP", "expose": True,
               "app_protocol": "http", "host": HOST}],
    "resources": {"cpu_request": 0.5, "cpu_limit": 0.5,
                  "memory_request": 1000, "memory_limit": 1000,
                  "ephemeral_storage_request": 2000, "ephemeral_storage_limit": 2000},
    "replicas": 1,
})
print(SVC, app.data.application_id)

iris-sk-fastapi-svc wlxvenlox1ku0hgvoab1nxan


## 6. Call the model

- Poll `/health` first — the pod has to pull the image and download the model into `$MODEL_PATH`

In [15]:
deadline = time.time() + READY_TIMEOUT
while time.time() < deadline:
    try:
        if requests.get(f"https://{HOST}/health", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f"{SVC} not ready after {READY_TIMEOUT}s")

print(SVC, "ready")

iris-sk-fastapi-svc ready


In [21]:
r = requests.post(f"https://{HOST}/infer",
                  json={"input_0": [[5.1, 3.5, 1.4, 0.2]]}, timeout=30)
print(r.status_code, r.text)

200 {"output_0":[0]}


The same call as a copy-pasteable `curl`, for testing outside the notebook:

In [22]:
print(f'''curl -X POST https://{HOST}/infer \\
  -H 'Content-Type: application/json' \\
  -d '{{"input_0": [[5.1, 3.5, 1.4, 0.2]]}}' ''')

curl -X POST https://iris-sk-fastapi-svc-seldon-tf-8000.tfy-usea1-ctl.devtest.truefoundry.tech/infer \
  -H 'Content-Type: application/json' \
  -d '{"input_0": [[5.1, 3.5, 1.4, 0.2]]}' 


## Cleanup
- service, local package, model file, image (local **and** Docker Hub)
- model version stays — a re-run logs a new one
- Docker Hub delete needs `$DOCKERHUB_PAT`

In [ ]:
# service
APP_ID = app.data.application_id
try:
    tfy.applications.delete(id=APP_ID)
except Exception:
    tfy.applications.cancel_deployment(
        id=APP_ID, deployment_id=tfy.applications.get(id=APP_ID).data.last_deployment_id)
    tfy.applications.delete(id=APP_ID)
print("deleted", SVC)

# local package, model file, image
!rm -rf {SVC} {MODEL_FILE}
!docker rmi -f {IMAGE}

# Docker Hub copy of the tag
jwt = requests.post("https://hub.docker.com/v2/users/login/",
    json={"username": DOCKER_USER, "password": os.environ["DOCKERHUB_PAT"]}).json()["token"]
r = requests.delete(f"https://hub.docker.com/v2/repositories/{DOCKER_USER}/{SVC}/tags/{TAG}/",
    headers={"Authorization": f"JWT {jwt}"})
print(r.status_code)  # 204 = gone